In [ ]:
from google.colab import drive
# 1. Montar o Google Drive
mount_drive = True
if mount_drive is True:
    drive.mount('/content/drive')

# 2. Instalar dependências necessárias (conforme o enunciado)
install_dependencies = True
if install_dependencies is True:
    !pip install torch-fidelity datasets -q



# No início do Notebook (antes do treino)
unzip_data = True
if unzip_data is True:
    !unzip -q /content/zip_data.zip -d /content/dataset_local

In [ ]:
# Bibliotecas nativas do Python
from __future__ import annotations
import sys
import json
import csv
import pickle
import random
from pathlib import Path
from collections import Counter

# Processamento de Dados e Visualização
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# PyTorch e Torchvision
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T 
from torchvision.utils import save_image, make_grid

# HuggingFace Datasets e Métricas
from datasets import Dataset as HFDataset, DatasetDict, Features, Image, ClassLabel, load_dataset
from torch_fidelity import calculate_metrics

# Define device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")


### CONFIG

In [ ]:
from pathlib import Path

# Caminhos dos Dados
PROJECT_ROOT = Path("/content/drive/MyDrive/IAG_PROJ_1")

KAGGLE_ROOT = Path("/content/dataset_local") / "data" / "archive"
EXPORT_ROOT  = Path("/content/dataset_local") / "data"
CSV_PATH_20 = Path("/content/dataset_local") / "assets" / "training_20_percent.csv"

# Dataset setup
INDEX_COLUMN = 'train_id_original' #warning if using colab kernel on vscode you need to put the files on your google drive and link this notebook to it.
TRAIN_FRACTION = 1.0  # Example: 0.5 means half of train split
IMAGE_SIZE = 32
NUM_WORKERS = 0

# Hiperparâmetros gerais
SEED = 42 # 42 -- 123
BATCH_SIZE = 64

# MODEL VARIABLES
N_EPOCHS = 50
WEIGHT_DECAY = 1e-4

# VAE
VAE_BETA = 0.7
VAE_LR = 1e-3
VAE_LATENT_DIM = 16 # 16 -- 128
VAE_OPTIM = "ADAM" # ADAM -- ADAMW

# GAN
GAN_LATENT_DIM = 100  # 16 -- 100
GAN_LR = 2e-4
GAN_BETA1 = 0.5       # 0.5 -- 0.9
GAN_BETA_TEXT = "5" if GAN_BETA1 == 0.5 else "9"

# Pastas específicas por arquitetura
RESULTS_BASE_DIR = PROJECT_ROOT / "results"
PASTA_REAIS_FID = Path("/content/dataset_local") / "data" / "real_fid_samples"

VAE_RESULTS_DIR = RESULTS_BASE_DIR / "vae"
VAE_RESULT_PATH = VAE_RESULTS_DIR / f"run_opt{VAE_OPTIM}_Latent{VAE_LATENT_DIM}_SEED{SEED}"
VAE_PASTA_FID = VAE_RESULT_PATH / "fid_samples"


GAN_RESULTS_DIR = RESULTS_BASE_DIR / "gan"
GAN_RESULT_PATH = GAN_RESULTS_DIR / f"run_beta_{GAN_BETA_TEXT}_Latent{GAN_LATENT_DIM}_SEED{SEED}"
GAN_PASTA_FID = GAN_RESULT_PATH / "fid_samples"

DIFFUSION_RESULTS_DIR = RESULTS_BASE_DIR / "diffusion"

CAMINHO_VAE_FID_JSON = VAE_RESULT_PATH / "fid_metrics.json"
CAMINHO_GAN_FID_JSON = GAN_RESULT_PATH / "fid_metrics.json"

# Create base folders
KAGGLE_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
VAE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GAN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIFFUSION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Define seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### ARTBEMCH_LOCAL_DATASET

In [ ]:
KAGGLE_SOURCE_NAMES = {"kaggle", "local", "artbench10"}

def dataset_source_name(dataset_source, default_source="hf"):
    src = str(dataset_source).strip().lower()
    if not src:
        return str(default_source).strip().lower()
    return src

def _get_pickle_value(obj, key):
    if key in obj:
        return obj[key]
    bkey = key.encode("utf-8")
    if bkey in obj:
        return obj[bkey]
    raise KeyError(f"Missing key '{key}' in pickle object")

def _resolve_kaggle_paths(kaggle_root):
    root = Path(kaggle_root)
    csv_path = root / "ArtBench-10.csv"
    batch_dir = root / "artbench-10-python" / "artbench-10-batches-py"
    return root, csv_path, batch_dir

def load_kaggle_artbench10_splits(kaggle_root):
    root, csv_path, batch_dir = _resolve_kaggle_paths(kaggle_root)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Kaggle CSV not found: {csv_path}. "
            "Expected the original ArtBench-10 folder structure."
        )
    if not batch_dir.exists():
        raise FileNotFoundError(
            f"Kaggle CIFAR batches not found: {batch_dir}. "
            "Expected ArtBench-10/artbench-10-python/artbench-10-batches-py"
        )

    with open(batch_dir / "meta", "rb") as f:
        meta = pickle.load(f)

    styles = _get_pickle_value(meta, "styles")

    if not isinstance(styles, list) or len(styles) == 0:
        raise ValueError(f"Could not read class names from {batch_dir / 'meta'}")
    
    styles = [str(s).strip() for s in styles]
    style_to_id = {name: i for i, name in enumerate(styles)}

    csv_label_ids = {"train": {}, "test": {}}
    
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        required = {"split", "label", "cifar_index"}
        missing = required.difference(set(reader.fieldnames or []))
        if missing:
            raise ValueError(f"CSV is missing required columns {sorted(missing)}: {csv_path}")

        for row in reader:
            split = str(row.get("split", "")).strip().lower()
            if split not in csv_label_ids:
                continue

            label_name = str(row.get("label", "")).strip()
            if label_name not in style_to_id:
                raise ValueError(
                    f"Unknown label '{label_name}' in {csv_path}. "
                    f"Known labels: {styles}"
                )

            try:
                idx = int(row.get("cifar_index"))
            except Exception as exc:
                raise ValueError(f"Invalid cifar_index '{row.get('cifar_index')}' in {csv_path}") from exc

            csv_label_ids[split][idx] = int(style_to_id[label_name])

    def _load_batch(path):
        with open(path, "rb") as f:
            batch = pickle.load(f)
        data = np.asarray(_get_pickle_value(batch, "data"), dtype=np.uint8)
        labels = np.asarray(_get_pickle_value(batch, "labels"), dtype=np.int64)
        if data.ndim != 2 or data.shape[1] != 3072:
            raise ValueError(f"Unexpected data shape in {path}: {data.shape}")
        images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, labels

    train_images_chunks = []
    train_labels_chunks = []
    for batch_idx in range(1, 6):
        images, labels = _load_batch(batch_dir / f"data_batch_{batch_idx}")
        train_images_chunks.append(images)
        train_labels_chunks.append(labels)
    train_images = np.concatenate(train_images_chunks, axis=0)
    train_labels_raw = np.concatenate(train_labels_chunks, axis=0)
    test_images, test_labels_raw = _load_batch(batch_dir / "test_batch")

    def _labels_from_csv(split, n, labels_raw):
        ids = csv_label_ids[split]
        out = np.full((n,), -1, dtype=np.int64)
        for idx, label_id in ids.items():
            if idx < 0 or idx >= n:
                raise ValueError(
                    f"CSV {split} index {idx} out of bounds for {n} samples ({csv_path})"
                )
            out[idx] = int(label_id)
        missing = int(np.sum(out < 0))
        if missing > 0:
            raise ValueError(
                f"CSV {csv_path} is missing {missing} labels for split '{split}'."
            )
        mismatches = int(np.sum(out != labels_raw))
        if mismatches > 0:
            raise ValueError(
                f"CSV labels and batch labels disagree for {mismatches} samples in split '{split}'."
            )
        return out

    train_labels = _labels_from_csv("train", train_images.shape[0], train_labels_raw)
    test_labels = _labels_from_csv("test", test_images.shape[0], test_labels_raw)

    features = Features({
        "image": Image(),
        "label": ClassLabel(names=styles),
    })

    train_ds = HFDataset.from_dict(
        {
            "image": [train_images[i] for i in range(train_images.shape[0])],
            "label": train_labels.tolist(),
        },
        features=features,
    )
    test_ds = HFDataset.from_dict(
        {
            "image": [test_images[i] for i in range(test_images.shape[0])],
            "label": test_labels.tolist(),
        },
        features=features,
    )

    print(f"Dataset source: kaggle root='{root}'")
    return DatasetDict(train=train_ds, test=test_ds)

def resolve_dataset_splits(dataset_id, seed=42, dataset_source="hf", kaggle_root="ArtBench-10", default_source="hf"):
    source = dataset_source_name(dataset_source, default_source=default_source)
    if source in KAGGLE_SOURCE_NAMES:
        return load_kaggle_artbench10_splits(kaggle_root)
    if source != "hf":
        raise ValueError(
            f"Invalid dataset_source='{dataset_source}'. Use 'kaggle' or 'hf'."
        )

    ds = load_dataset(dataset_id)
    if isinstance(ds, dict) and not isinstance(ds, DatasetDict):
        ds = DatasetDict(ds)

    if "train" not in ds:
        first = next(iter(ds.keys()))
        spl = ds[first].train_test_split(test_size=1 / 6, seed=seed)
        ds = DatasetDict(train=spl["train"], test=spl["test"])
    elif "test" not in ds:
        spl = ds["train"].train_test_split(test_size=1 / 6, seed=seed)
        ds = DatasetDict(train=spl["train"], test=spl["test"])

    print(f"Dataset source: hf dataset_id='{dataset_id}'")
    return ds


### DATALOADER

In [ ]:
def get_transforms(image_size):
    """Retorna as transformações padrão para as imagens."""
    return T.Compose([
        T.Resize(image_size, interpolation=T.InterpolationMode.BILINEAR),
        T.CenterCrop(image_size),
        T.ToTensor(),  # converte para [0,1]
    ])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform else img
        return x, y, real_idx

### IO

In [ ]:
def safe_num_workers(requested: int) -> int:
    # Avoid notebook multiprocessing pickling issues on macOS/ipykernel.
    if "ipykernel" in sys.modules and int(requested) > 0:
        print("Notebook kernel detected: forcing num_workers=0 for DataLoader stability.")
        return 0
    return int(requested)

def make_subset_indices(n_total: int, fraction: float, seed: int = 42):
    n_keep = max(1, int(round(n_total * fraction)))
    g = np.random.RandomState(seed)
    idx = np.arange(n_total)
    g.shuffle(idx)
    return idx[:n_keep].tolist()

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"training.csv not found: {csv_path}\n"
            "Generate it first with scripts/generate_training_csv.py"
        )

    ids = []
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        r = csv.DictReader(f)
        if index_column not in (r.fieldnames or []):
            raise ValueError(
                f"Column {index_column!r} not present in {csv_path}. "
                f"Available: {r.fieldnames}"
            )
        for row in r:
            v = str(row.get(index_column, "")).strip()
            if v == "":
                continue
            ids.append(int(v))

    if len(ids) == 0:
        raise ValueError(f"No ids found in {csv_path} column {index_column!r}")
    return ids

def export_split_to_folder(
    loader: DataLoader,
    class_names: list[str],
    out_dir: Path,
    max_images: int | None = 500,
):
    out_dir = Path(out_dir)
    img_dir = out_dir / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    saved = 0

    for x, y, idx in loader:
        b = x.shape[0]
        for i in range(b):
            if max_images is not None and saved >= max_images:
                break

            label_id = int(y[i].item())
            label_name = class_names[label_id]
            src_idx = int(idx[i].item())

            file_name = f"img_{saved:06d}_label{label_id:02d}_idx{src_idx:06d}.png"
            path = img_dir / file_name
            save_image(x[i], path)

            rows.append({
                'file_name': file_name,
                'label_id': label_id,
                'label_name': label_name,
                'source_index': src_idx,
            })
            saved += 1

        if max_images is not None and saved >= max_images:
            break

    csv_path = out_dir / 'metadata.csv'
    with open(csv_path, 'w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['file_name', 'label_id', 'label_name', 'source_index'])
        w.writeheader()
        w.writerows(rows)

    print(f'Exported {saved} images to: {img_dir}')
    print(f'Metadata CSV: {csv_path}')

def save_experiment_results(run_dir: Path, model, history: list, config_dict: dict, test_metrics: dict = None):
    """
    Guarda o modelo, o gráfico de loss, o histórico em CSV, configs e métricas finais.
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n💾 A guardar resultados da experiência em:\n -> {run_dir}")
    
    # 1. Guardar Configurações (JSON)
    with open(run_dir / "config.json", "w") as f:
        json.dump(config_dict, f, indent=4)
        
    # 2. Guardar Métricas Finais de Teste (JSON)
    if test_metrics:
        with open(run_dir / "test_metrics.json", "w") as f:
            json.dump(test_metrics, f, indent=4)
        
    # 3. Guardar Pesos do Modelo (.pth)
    torch.save(model.state_dict(), run_dir / "model.pth")
    
    # 4. Guardar Histórico (CSV)
    df_history = pd.DataFrame(history)
    df_history.to_csv(run_dir / "history.csv", index=False)
    
    # 5. Desenhar e Guardar Gráfico de Loss
    plt.figure(figsize=(10, 5))
    plt.plot(df_history['train_loss'], label='Loss Total', color='black', linewidth=2)
    plt.plot(df_history['train_recon_bce'], label='Reconstrução (BCE)', color='blue', linestyle='--')
    plt.plot(df_history['train_kl'], label='KL Divergence', color='red', linestyle='--')
    
    plt.title("Curvas de Treino - VAE", fontsize=14)
    plt.xlabel("Épocas")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.savefig(run_dir / "loss_curve.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✅ Guardado com sucesso!")

# Define Number os workers
EFFECTIVE_NUM_WORKERS = safe_num_workers(NUM_WORKERS)

### METRICS

In [ ]:
def evaluate_vae(model, loader, device, beta=0.7):
    model.eval()
    tl, tr, tk, tm, ta, n = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x, _, _ in loader:
            x = x.to(device)
            xhat, mu, logvar = model(x)
            b = x.size(0)
            loss, recon, kl = vae_loss(xhat, x, mu, logvar, beta=beta)
            
            tl += loss.item() * b
            tr += recon.item() * b
            tk += kl.item() * b
            tm += F.mse_loss(xhat, x, reduction='sum').item()
            ta += F.l1_loss(xhat, x, reduction='sum').item()
            n += b

    numel = x[0].numel()
    return {'loss': tl/n, 'recon_bce': tr/n, 'kl': tk/n, 'mse': tm/(n*numel), 'mae': ta/(n*numel)}

def interpolacao_latente_vae(model, img_a, img_b, device, steps=10, save_path=None):
    """Cria transição suave entre a img_a e a img_b no espaço latente."""
    model.eval()
    with torch.no_grad():
        img_a = img_a.unsqueeze(0).to(device)
        img_b = img_b.unsqueeze(0).to(device)
        
        mu_a, logvar_a = model.encode(img_a)
        z_a = model.reparameterize(mu_a, logvar_a)
        
        mu_b, logvar_b = model.encode(img_b)
        z_b = model.reparameterize(mu_b, logvar_b)
        
        alphas = np.linspace(0, 1, steps)
        imagens_geradas = []
        
        for alpha in alphas:
            z_interp = (1.0 - alpha) * z_a + alpha * z_b
            img_gerada = model.decode(z_interp)
            imagens_geradas.append(img_gerada.squeeze(0).cpu().permute(1, 2, 0).numpy())
            
    fig, axes = plt.subplots(1, steps, figsize=(15, 3))
    fig.suptitle("Interpolação no Espaço Latente (A -> B)", fontsize=16)
    for i, ax in enumerate(axes):
        ax.imshow(imagens_geradas[i])
        ax.axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

def gerar_amostras_fid(model, num_samples, batch_size, device, output_dir):
    """Gera amostras e guarda numa pasta para cálculo do FID/KID."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    model.eval()
    amostras_geradas = 0
    print(f"\nA gerar {num_samples} imagens para FID em: {output_dir}")
    
    with torch.no_grad(), tqdm(total=num_samples) as pbar:
        while amostras_geradas < num_samples:
            n_gerar = min(batch_size, num_samples - amostras_geradas)
            batch_imagens = model.sample(n_gerar, device)
            
            for i in range(n_gerar):
                nome_ficheiro = output_dir / f"fake_{amostras_geradas:05d}.png"
                save_image(batch_imagens[i], nome_ficheiro)
                amostras_geradas += 1
                pbar.update(1)

def mostrar_reconstrucoes(modelo, test_loader, device, num_imagens=8, save_path=None):
    """
    Compara imagens reais do test_loader com as reconstruções geradas pelo VAE.
    """
    modelo.eval()
    
    # Extrair o primeiro batch (Imagens estão no índice 0)
    batch = next(iter(test_loader))
    imagens_reais = batch[0][:num_imagens].to(device)
    
    with torch.no_grad():
        imagens_geradas, _, _ = modelo(imagens_reais)
    
    # Mover para CPU e ajustar os eixos para o Matplotlib
    imagens_reais = imagens_reais.cpu().permute(0, 2, 3, 1).numpy()
    imagens_geradas = imagens_geradas.cpu().permute(0, 2, 3, 1).numpy()
    
    fig, axes = plt.subplots(2, num_imagens, figsize=(15, 4))
    fig.suptitle("Original (Cima) vs Reconstrução VAE (Baixo)", fontsize=16)
    
    for i in range(num_imagens):
        axes[0, i].imshow(imagens_reais[i])
        axes[0, i].axis('off')
        
        axes[1, i].imshow(imagens_geradas[i])
        axes[1, i].axis('off')
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"✅ Reconstruções guardadas em: {save_path}")
        
    plt.show()

def gerar_grelha_amostras(modelo, device, num_samples=16, save_path=None):
    """
    Gera novas imagens a partir de ruído aleatório no espaço latente e mostra-as numa grelha.
    """
    modelo.eval()
    with torch.no_grad():
        # O método sample já faz o torch.randn internamente!
        amostras = modelo.sample(num_samples, device)
        
        # Converter para visualização no Matplotlib
        amostras = amostras.cpu().permute(0, 2, 3, 1).numpy()
        
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle("Artes Geradas do Zero (Random Latent Sampling)", fontsize=16)
    
    for i, ax in enumerate(axes.flat):
        if i < num_samples: # Proteção caso peçam um número diferente de 16
            ax.imshow(amostras[i])
        ax.axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"✅ Grelha de amostras guardada em: {save_path}")
        
    plt.show()

def extrair_amostras_reais_fid(loader, num_samples, output_dir):
    """Extrai imagens reais do DataLoader para uma pasta (para o cálculo do FID)."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    amostras_salvas = 0
    print(f"\n🖼️ A extrair {num_samples} imagens REAIS para FID em: {output_dir}")
    
    with torch.no_grad(), tqdm(total=num_samples) as pbar:
        for x, _, _ in loader:
            batch_size = x.size(0)
            for i in range(batch_size):
                if amostras_salvas >= num_samples:
                    print("✅ Extração de imagens reais concluída!")
                    return
                
                nome_ficheiro = output_dir / f"real_{amostras_salvas:05d}.png"
                save_image(x[i], nome_ficheiro)
                amostras_salvas += 1
                pbar.update(1)


def show_batch_grid(loader, class_names, n_images=36, nrow=6, title='Sample Grid'):
    x, y, idx = next(iter(loader))
    x = x[:n_images]
    y = y[:n_images]

    grid = make_grid(x, nrow=nrow, padding=2)
    np_img = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(np_img)
    plt.axis('off')
    plt.title(title)
    plt.show()

    # Print labels for quick inspection
    labels_str = [class_names[int(v)] for v in y]
    print('Labels:', labels_str)

### DATASET

In [ ]:
hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]

print("Train size:", len(train_hf))
print("Columns   :", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)

print("Num classes:", num_classes)
print("Class names:", class_names)

In [ ]:
# Class distribution summary
train_counts = Counter(train_hf["label"])

print("\nTrain class distribution:")
for cid, name in enumerate(class_names):
    print(f"  {cid:2d} | {name:>15s} | {train_counts.get(cid, 0):6d}")

In [ ]:
transform = get_transforms(IMAGE_SIZE)

train_indices = make_subset_indices(len(train_hf), TRAIN_FRACTION, seed=SEED)

train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_indices)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train dataset length (after fraction):", len(train_ds))
print("Train batches                        :", len(train_loader))

In [ ]:
train_ids_from_csv = load_ids_from_training_csv(CSV_PATH_20, index_column=INDEX_COLUMN)

print('Loaded ids:', len(train_ids_from_csv))
print('First 10 ids:', train_ids_from_csv[:10])

# Build a train dataset/loader using exactly those IDs
train_ds_from_csv = HFDatasetTorch(train_hf, transform=transform, indices=train_ids_from_csv)
train_loader_from_csv = DataLoader(
    train_ds_from_csv,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('Subset train dataset length:', len(train_ds_from_csv))
print('Subset train batches      :', len(train_loader_from_csv))


In [ ]:
show_batch_grid(train_loader, class_names, n_images=36, nrow=6, title='ArtBench-10 Train Samples')

In [ ]:
export_split_to_folder(train_loader, class_names, EXPORT_ROOT / 'train_subset', max_images=500)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )

        flattened_dim = 128 * 4 * 4
        
        self.fc_mu = nn.Linear(flattened_dim, latent_dim)
        self.fc_logvar = nn.Linear(flattened_dim, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 128 * 4 * 4)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x) # Usa o nome correto
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        # Usa o nome correto e altera o 7x7 para 4x4
        h = self.decoder_input(z).view(-1, 128, 4, 4) 
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar
    
    def sample(self, num_samples, device):
        z = torch.randn(num_samples, self.latent_dim).to(device)
        z = self.decoder_input(z).view(-1, 128, 4, 4)
        samples = self.decoder(z)
        return samples

def vae_loss(xhat, x, mu, logvar, beta=0.7):
    recon_loss = F.binary_cross_entropy(xhat, x, reduction='sum') / x.shape[0]
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]
    loss = recon_loss + beta * kl_loss
    return loss, recon_loss, kl_loss

def train_vae(model, loader, optimizer, device, epochs=20, beta=0.7):
    model.train()
    hist = []
    for ep in range(epochs):
        tl, tr, tk = 0.0, 0.0, 0.0
        for x, _, _ in tqdm(loader, leave=False):
            x = x.to(device)

            optimizer.zero_grad()
            xhat, mu, logvar = model(x)
            loss, recon, kl = vae_loss(xhat, x, mu, logvar, beta=beta)
            
            loss.backward()
            optimizer.step()

            tl += loss.item() * x.size(0)
            tr += recon.item() * x.size(0)
            tk += kl.item() * x.size(0)
            
        n = len(loader.dataset)
        hist.append({'train_loss': tl/n, 'train_recon_bce': tr/n, 'train_kl': tk/n})
        print(f'Epoch {ep+1}/{epochs} | train_loss={tl/n:.4f} train_recon={tr/n:.4f} train_kl={tk/n:.4f}')
    return hist

### VAE

In [ ]:
# # Recriar o test_loader para garantir que ele entrega (imagem, label, id)
# test_dataset = HFDatasetTorch(hf_ds["test"], transform=transform)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Inicializar o Modelo e o Otimizador
# model_vae = VAE(latent_dim=VAE_LATENT_DIM).to(device)

# if VAE_OPTIM == "ADAM":
#     optimizer = torch.optim.Adam(model_vae.parameters(), lr=VAE_LR)
# else:
#     optimizer = torch.optim.AdamW(model_vae.parameters(), lr=VAE_LR, weight_decay=1e-4)

# # Executar o Treino
# print("\nA iniciar o treino do VAE no dataset ArtBench (20%)...")
# hist_vae = train_vae(
#     model=model_vae, 
#     loader=train_loader, 
#     optimizer=optimizer, 
#     device=device,       
#     epochs=N_EPOCHS, 
#     beta=VAE_BETA
# )
# print("Treino concluído!")

### EVALUATE

In [ ]:
# # AVALIAÇÃO DO MODELO 

# print("\n📊 A avaliar o modelo nos dados de teste...")
# metrics_vae = evaluate_vae(
#     model=model_vae, 
#     loader=test_loader, 
#     device=device, 
#     beta=VAE_BETA
# )

# print('\n=== Resultados Finais do VAE ===')
# print(f"Loss Total: {metrics_vae['loss']:.4f}")
# print(f"Reconstrução (BCE): {metrics_vae['recon_bce']:.4f}")
# print(f"KL Divergence: {metrics_vae['kl']:.4f}")

# configs_usadas = {
#     "model": "VAE",
#     "latent_dim": VAE_LATENT_DIM,
#     "beta": VAE_BETA,
#     "epochs": N_EPOCHS,
#     "batch_size": BATCH_SIZE,
#     "seed": SEED
# }

# save_experiment_results(
#     run_dir=VAE_RESULT_PATH,
#     model=model_vae,
#     history=hist_vae,
#     config_dict=configs_usadas,
#     test_metrics=metrics_vae
# )

# # GERAÇÕES VISUAIS E FID
# print("\n🎨 A gerar interpolação visual...")
# batch = next(iter(test_loader))
# imagens_reais = batch[0]
# img_A = imagens_reais[0] 
# img_B = imagens_reais[1] 

# # Guarda o gráfico de interpolação!
# interpolacao_latente_vae(model_vae, img_A, img_B, device, steps=8, save_path=VAE_RESULT_PATH / "interpolacao.png")

# print("\n🚀 A preparar amostras para FID...")

# gerar_amostras_fid(model_vae, 5000, BATCH_SIZE, device, VAE_PASTA_FID)

In [ ]:
# print("\n🖼️ A visualizar reconstruções...")
# mostrar_reconstrucoes(
#     modelo=model_vae, 
#     test_loader=test_loader, 
#     device=device,
#     num_imagens=8,
#     save_path=VAE_RESULT_PATH / "reconstrucoes.png"
# )

In [ ]:
# print("\n✨ A gerar artes completamente novas...")
# gerar_grelha_amostras(
#     modelo=model_vae, 
#     device=device,
#     num_samples=16,
#     save_path=VAE_RESULT_PATH / "amostras_aleatorias.png"
# )

In [ ]:
# Extrair 5000 imagens reais do train_loader
# extrair_amostras_reais_fid(
#     loader=train_loader, 
#     num_samples=5000, 
#     output_dir=PASTA_REAIS_FID
# )

In [ ]:
# print("A iniciar o cálculo matemático do FID e KID...")
# print(f"-> Pasta 1 (Reais): {PASTA_REAIS_FID}")
# print(f"-> Pasta 2 (Geradas): {VAE_PASTA_FID}")

# # 1. Chamar a API do torch-fidelity internamente
# metricas = calculate_metrics(
#     input1=str(PASTA_REAIS_FID),  
#     input2=str(VAE_PASTA_FID),        
#     cuda=False,
#     isc=False,                    
#     fid=True,                     
#     kid=True,                     
#     verbose=True                  
# )

# print("\n🎉 === RESULTADOS FINAIS === 🎉")
# print(f"FID (Fréchet Inception Distance): {metricas['frechet_inception_distance']:.4f}")
# print(f"KID (Kernel Inception Distance) : {metricas['kernel_inception_distance_mean']:.5f}")

# with open(CAMINHO_VAE_FID_JSON, "w") as f:
#     json.dump(metricas, f, indent=4)

# print(f"\n✅ Relatório de métricas guardado com sucesso em:\n -> {CAMINHO_VAE_FID_JSON}")

### GANs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt

class DCGenerator(nn.Module):
    def __init__(self, latent_dim=100, image_channels=3, ngf=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, image_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        z = z.view(z.size(0), self.latent_dim, 1, 1)
        return self.net(z)

class DCDiscriminator(nn.Module):
    def __init__(self, image_channels=3, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(image_channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)

def init_dcgan_weights(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

def save_checkpoint(generator, discriminator, history, checkpoint_path, latent_dim, channels, image_size):
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'generator': generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'history': history,
            'config': {
                'latent_dim': latent_dim,
                'channels': channels,
                'image_size': image_size,
            },
        },
        checkpoint_path,
    )
    print('Saved checkpoint to', checkpoint_path)

@torch.no_grad()
def load_dcgan_generator_for_inference(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    cfg = ckpt['config']
    generator = DCGenerator(latent_dim=cfg['latent_dim'], image_channels=cfg['channels']).to(device)
    generator.load_state_dict(ckpt['generator'])
    generator.eval()
    return generator, cfg, ckpt.get('history', None)

def train_gan(generator, discriminator, loader, latent_dim, epochs=20, lr=2e-4, device='cpu'):
    criterion = nn.BCELoss()

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    history = {'g_loss': [], 'd_loss': []}
    generator.train()
    discriminator.train()

    for epoch in range(epochs):
        g_running = 0.0
        d_running = 0.0
        n_batches = 0

        for real, _, _ in tqdm(loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False):
            real = real.to(device)
            bs = real.size(0)

            real_targets = torch.ones(bs, 1, device=device)
            fake_targets = torch.zeros(bs, 1, device=device)

            # TODO START - Discriminator update
            # 1) Reset discriminator gradients for the new mini-batch.
            opt_d.zero_grad()
            
            # 2) Measure how well it recognizes real images as real.
            d_loss_real = criterion(discriminator(real), real_targets)
            
            # 3) Sample latent noise and generate a fake batch.
            z = torch.randn(bs, latent_dim, device=device)
            fake = generator(z)
            
            # 4) Measure how well it recognizes fake images as fake.
            # Crucial: using .detach() so we don't backprop through the generator here
            d_loss_fake = criterion(discriminator(fake.detach()), fake_targets)
            
            # 5) Combine both real/fake discriminator losses.
            d_loss = d_loss_real + d_loss_fake
            
            # 6) Backpropagate discriminator loss and update discriminator weights.
            d_loss.backward()
            opt_d.step()
            # TODO END

            if d_loss is None:
                raise NotImplementedError('Implement Discriminator update in train_gan.')

            # TODO START - Generator update
            # 1) Reset generator gradients for the new mini-batch.
            opt_g.zero_grad()
            
            # 2) Sample fresh latent noise and generate a fake batch.
            z = torch.randn(bs, latent_dim, device=device)
            fake = generator(z)
            
            # 3) Evaluate how convincing those fake images look to the discriminator.
            preds = discriminator(fake)
            
            # 4) Compute generator loss so fake images are pushed toward "real" predictions.
            g_loss = criterion(preds, real_targets)
            
            # 5) Backpropagate generator loss and update generator weights.
            g_loss.backward()
            opt_g.step()
            # TODO END

            if g_loss is None:
                raise NotImplementedError('Implement Generator update in train_gan.')

            # TODO START - Bookkeeping
            g_running += g_loss.item()
            d_running += d_loss.item()
            n_batches += 1
            # TODO END

        history['g_loss'].append(g_running / max(n_batches, 1))
        history['d_loss'].append(d_running / max(n_batches, 1))

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"D loss: {history['d_loss'][-1]:.4f} | "
            f"G loss: {history['g_loss'][-1]:.4f}"
        )

    return history

In [ ]:
def plot_gan_losses(history, title='GAN losses'):
    plt.figure(figsize=(7, 4))
    plt.plot(history['d_loss'], label='Discriminator loss')
    plt.plot(history['g_loss'], label='Generator loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
@torch.no_grad()
def interpolacao_latente_gan(generator, latent_dim, steps=8, device='cpu', save_path=None):
    """Faz uma interpolação linear entre dois pontos no espaço latente."""
    generator.eval()
    
    # TODO START SOLVED
    # 1) sample z0 and z1
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    
    # 2) interpolate with alpha in [0, 1]
    alphas = np.linspace(0, 1, steps)
    
    # 3) generate fake images (Iterando pelos alphas)
    imagens_geradas = []
    for alpha in alphas:
        z_interp = (1.0 - alpha) * z0 + alpha * z1
        fake = generator(z_interp)
        
        # Desnormalizar
        fake = (fake + 1) / 2.0
        imagens_geradas.append(fake.squeeze(0).cpu().permute(1, 2, 0).numpy())
    # TODO END

    fig, axes = plt.subplots(1, steps, figsize=(15, 3))
    fig.suptitle("Interpolação no Espaço Latente DCGAN (A -> B)", fontsize=16)
    for i, ax in enumerate(axes):
        ax.imshow(imagens_geradas[i])
        ax.axis('off')
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"✅ Interpolação guardada em: {save_path}")
    plt.show()

def gerar_amostras_fid_gan(generator, num_samples, batch_size, latent_dim, device, output_dir):
    """Gera amostras e guarda numa pasta para cálculo do FID/KID."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    generator.eval()
    amostras_geradas = 0
    print(f"\n🚀 A preparar {num_samples} amostras FID para a GAN em: {output_dir}")
    
    with torch.no_grad(), tqdm(total=num_samples) as pbar:
        while amostras_geradas < num_samples:
            n_gerar = min(batch_size, num_samples - amostras_geradas)
            z = torch.randn(n_gerar, latent_dim, device=device)
            fake_images = generator(z)
            
            # CRUCIAL: Desnormalizar para guardar corretamente!
            fake_images = (fake_images + 1) / 2.0 
            
            for i in range(n_gerar):
                nome_ficheiro = output_dir / f"fake_gan_{amostras_geradas:05d}.png"
                save_image(fake_images[i], nome_ficheiro)
                amostras_geradas += 1
                pbar.update(1)

@torch.no_grad()
def run_inference(generator, latent_dim, num_samples=16, seed=123, device='cpu', save_path=None):
    """Gera uma grelha de imagens a partir de ruído aleatório."""
    generator.eval()
    
    # TODO START SOLVED
    # 1) set torch seed
    torch.manual_seed(seed)
    
    # 2) sample z
    z = torch.randn(num_samples, latent_dim, device=device)
    
    # 3) generate fake images
    fake = generator(z)
    # TODO END

    # CRUCIAL: Desnormalizar para plotar cores reais
    fake = (fake + 1) / 2.0 
    
    # Plot e Save (Substitui o show_image_grid para podermos guardar)
    fake_np = fake.cpu().permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle("Artes Geradas do Zero (DCGAN)", fontsize=16)
    
    for i, ax in enumerate(axes.flat):
        if i < num_samples:
            ax.imshow(fake_np[i])
        ax.axis('off')
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"✅ Grelha guardada em: {save_path}")
    plt.show()

def get_transforms_gan(image_size):
    """Transformações para a GAN: coloca os píxeis no intervalo [-1, 1]"""
    return T.Compose([
        T.Resize(image_size, interpolation=T.InterpolationMode.BILINEAR),
        T.CenterCrop(image_size),
        T.ToTensor(),  
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # <- A Magia do [-1, 1]
    ])

In [ ]:
## 🔄 Dataloader específico para a GAN (Normalização [-1, 1])
transform = get_transforms_gan(IMAGE_SIZE)
train_ds_gan = HFDatasetTorch(train_hf, transform=transform, indices=train_indices)

train_loader_gan = DataLoader(
    train_ds_gan,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
# Inicializar o Modelo
netG = DCGenerator(latent_dim=GAN_LATENT_DIM, image_channels=3).to(device)
netD = DCDiscriminator(image_channels=3).to(device)

# Inicializar pesos (Obrigatório para estabilidade)
netG.apply(init_dcgan_weights)
netD.apply(init_dcgan_weights)

# Executar o Treino
print(f"\n🚀 A iniciar o treino da DCGAN no dataset ArtBench por {N_EPOCHS} épocas...")
hist_gan = train_gan(
    generator=netG,
    discriminator=netD,
    loader=train_loader_gan, 
    latent_dim=GAN_LATENT_DIM,
    epochs=N_EPOCHS,
    lr=GAN_LR,
    device=device
)
print("Treino concluído!")

In [ ]:
print("\n💾 A guardar resultados da experiência GAN...")

# Guardar o modelo
save_checkpoint(
    generator=netG,
    discriminator=netD,
    history=hist_gan,
    checkpoint_path=GAN_RESULT_PATH / "gan_model.pt",
    latent_dim=GAN_LATENT_DIM,
    channels=3,
    image_size=IMAGE_SIZE
)

# Guardar configurações
configs_gan = {
    "model": "DCGAN",
    "latent_dim": GAN_LATENT_DIM,
    "epochs": N_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": GAN_LR,
    "seed": SEED
}
with open(GAN_RESULT_PATH / "config_gan.json", "w") as f:
    json.dump(configs_gan, f, indent=4)

# Gerar e guardar gráfico
plot_gan_losses(hist_gan, title='DCGAN Losses - ArtBench')
plt.savefig(GAN_RESULT_PATH / "loss_curve.png")

In [ ]:
print("\n✨ A gerar artes completamente novas (Random Latent Sampling)...")
run_inference(netG, GAN_LATENT_DIM, num_samples=16, device=device, save_path=GAN_RESULT_PATH / "amostras_aleatorias.png")

print("\n🎨 A gerar interpolação visual...")
interpolacao_latente_gan(netG, GAN_LATENT_DIM, steps=8, device=device, save_path=GAN_RESULT_PATH / "interpolacao.png")

# Executar a geração
gerar_amostras_fid_gan(netG, 5000, BATCH_SIZE, GAN_LATENT_DIM, device, GAN_PASTA_FID)

In [ ]:
print("A iniciar o cálculo matemático do FID e KID para a DCGAN...")
print(f"-> Pasta 1 (Reais): {PASTA_REAIS_FID}")
print(f"-> Pasta 2 (Geradas): {GAN_PASTA_FID}")

# Note: certifique-se que cuda=True se estiver a usar GPU no Colab
metricas_gan = calculate_metrics(
    input1=str(PASTA_REAIS_FID),  
    input2=str(GAN_PASTA_FID),        
    cuda=True,
    isc=False,                    
    fid=True,                     
    kid=True,                     
    verbose=True                  
)

print("\n🎉 === RESULTADOS FINAIS GAN === 🎉")
print(f"FID (Fréchet Inception Distance): {metricas_gan['frechet_inception_distance']:.4f}")
print(f"KID (Kernel Inception Distance) : {metricas_gan['kernel_inception_distance_mean']:.5f}")

with open(CAMINHO_GAN_FID_JSON, "w") as f:
    json.dump(metricas_gan, f, indent=4)

print(f"\n✅ Relatório de métricas da GAN guardado com sucesso em:\n -> {CAMINHO_GAN_FID_JSON}")